# RAG tuning

We work on the previous examples of RAG to attempt to improve answer accuracy.

We try different chunking strategies to see how answer quality is impacted.

In [69]:
from copy import deepcopy
import numpy as np
from IPython.display import HTML, Markdown, display
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.doug_blog_data import corpus

openai = OpenAI(api_key=key_for_provider('openai'))

## Search index + chunking

In [87]:
def no_chunking(row):
    yield row['description']

def chunk_by_overlap(row, chunk_size=65, overlap=20):
    """Chunk a document by token length with overlap."""
    tokens = row['description'].split()
    for i in range(0, len(tokens), chunk_size - overlap):
        yield " ".join(tokens[i:i + chunk_size])

def chunk_by_token_length(row, chunk_size=100):
    """Chunk a document by token length."""
    tokens = row['description'].split()
    for i in range(0, len(tokens), chunk_size):
        yield " ".join(tokens[i:i + chunk_size])

def chunk_with_title(chunk_fn):
    """Adapt a document chunker to accept a full corpus row."""
    def chunk_row(row):
        for chunk in chunk_fn(row):
            yield f"## {row['title']}\n\n{chunk}"
    return chunk_row

def top_n(vectors, query_vector, k=3):
    similarities = np.dot(vectors, query_vector)
    return np.argsort(similarities)[-k:][::-1]

class SearchIndex:
    def __init__(self, corpus, chunk_fn=chunk_by_overlap):
        self.corpus = corpus
        self.chunk_fn = chunk_with_title(chunk_fn)
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.all_chunks = []
        self.index = None

    def chunks(self):
        for _, row in self.corpus.iterrows():
            for chunk in self.chunk_fn(row):
                yield chunk

    def build_index(self):
        self.all_chunks = list(self.chunks())
        self.index = self.model.encode(
            [chunk for chunk in self.all_chunks],
            convert_to_numpy=True,
            show_progress_bar=True,
        )

    def search(self, query, k=5):
        query_vector = self.model.encode([query], convert_to_numpy=True)[0]
        return [self.all_chunks[i] for i in top_n(self.index, query_vector, k)]


index = SearchIndex(corpus)
index.build_index()
index.search("Best practices for BM25")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/413 [00:00<?, ?it/s]

['## Ugly hack to force BM25 to 0-1\n\ntips - [subscribe here](http://softwaredoug.kit.com)*',
 '## Can BM25 be a probability?\n\nrandom values and picking the best). It’s underrated and very practical. If you want to get fancier you can do [Bayesian optimization](https://haystackconf.com/us2022/talk-5/) on the params (a different notion of Bayes). You’re not calibrating trying to create good probabilities. Here L / E live on different scales. So you’re just picking magic numbers that predict the final score. What you lose: interpretability of these intermediate factors.',
 '## Can BM25 be a probability?\n\n0.1, 0.9) ``` There’s plenty to quibble with, and probably could be improved. Like other scoring system, we’ve found a heuristic that seems to work for the author. But probably not nearly as battle tested as the original BM25 scoring (remember its the 25th iteration!). The prior also doesn’t care about document frequency. The prior for 3 occurrences of “the” in a doc would be',
 '##

## Rewrite conversational requests

The rewrite is intentionally narrow. It removes the request to speak as Doug and produces a general information-retrieval question that should be easier to match against blog content.

In [88]:
answer_system_prompt = '''
You pretend to be search engine expert Doug Turnbull. Answer questions
in his voice using the supplied snippets from Doug's blog. Give Doug's
specific perspective, not a generic answer. If the snippets do not
support a claim, say that the blog context does not establish it.

If you cannot find information. Do not answer from your general knowledge. 
Just respond that you do not see any information on this topic.

Keep your answer as brief as possible (ie no more than 3 sentences)
'''

query_system_prompt = '''
The user is asking Doug Turnbull a question. Reformulate the user's
request as a general information-retrieval question to search Doug's blog.
Remove Doug's name and any request to answer in Doug's voice. Return only
the rewritten search query.
'''

judge_system_prompt = """
You compare two answers from a chatbot to see if they're identical. Your work aids in evaluation,
so its important to be careful and correct.

Criteria should be based on semantic overlap between the two:
MATCH - Main idea in ACTUAL and Main idea in EXPECTED are like 80% overlap
PARTIAL - ACTUAL part of answer, but many more components mentioned
NO-MATCH - ACTUAL nowhere to be found in answer

You're comparing an expected answer vs an actual. In the form:

## EXPECTED 
<...>

## ACTUAL:
<...>

(a) Your reasoning
(b) Your last line with: MATCH, NO-MATCH, PARTIAL
"""

class RAG:
    def __init__(self, corpus, client, chunk_fn=chunk_by_overlap):
        self.client = client
        self.search_index = SearchIndex(corpus, chunk_fn=chunk_fn)
        self.search_index.build_index()
        self.reset()

    def reset(self):
        self.context = [{'role': 'system', 'content': answer_system_prompt}]

    def query_rewrite(self, message):
        response = self.client.responses.create(
            model='gpt-3.5-turbo',
            input=[
                {'role': 'system', 'content': query_system_prompt},
                {'role': 'user', 'content': message},
            ],
        )
        query = response.output_text.strip()
        print("Rewritten: ", query)
        return query

    def chat(self, message):
        query = self.query_rewrite(message)
        results = self.search_index.search(query)
        # for result in results:
        #     print("---")
        #     print(result)
        snippets = '\n\n---\n\n'.join(snippet for snippet in results)
        context = deepcopy(self.context)
        context.append({'role': 'user', 'content': message})
        context.append({
            'role': 'user',
            'content': "Use these retrieved snippets from Doug's blog:\n\n" + snippets,
        })
        response = self.client.responses.create(
            model='gpt-3.5-turbo',
            input=context,
        )
        answer = response.output_text
        self.context.append({'role': 'user', 'content': message})
        self.context.append({'role': 'assistant', 'content': answer})
        return answer, query, results

    def judge(self, expected, actual):
        prompt = f"""
    
        ## EXPECTED
        {expected}
    
        ## ACTUAL
        {actual}
        """
    
        context = [
            {'role': 'system', 'content': judge_system_prompt},
            {'role': 'user', 'content': prompt}
        ]
        response = self.client.responses.create(
            model='gpt-3.5-turbo',
            input=context,
        )
        resp = response.output_text.strip().split()[-1]
        if 'NO-MATCH' in resp:
            return 'NO-MATCH'
        elif 'PARTIAL' in resp:
            return 'PARTIAL'
        elif 'MATCH' in resp:
            return 'MATCH'

In [89]:
rag = RAG(corpus, openai, chunk_fn=chunk_fn)
rag.judge("agentic search", "agentic search is one way which foobar is used")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

'PARTIAL'

In [101]:
from collections import Counter

message_w_answer = {
    'What is Bayesian BM25?': 
      'Bayesian BM25 calibrates BM25 scores to become meaningful probabilities by combining lexical and embedding scores in a probabilistic way.', 
    'According to Doug, In what context can grep be used to drive a search system?': 
       'in an agentic search system grep is valuable',
    'Whats a kind of semantic search besides embeddings?': "Besides embeddings, another type of semantic search method is hierarchical managed taxonomies.",
    "What does Doug observe boosts reasoning almost 35%?": "agentic reasoning",
    'How is search management useful?': 'it takes the pressure off algorithms to solve weird edge-case issues',
    'Why use pairwise evals over pointwise': 'Pairwise evals over pointwise can offer advantages like fewer per-decision errors' + 
    ', more precise results, and faster decisions due to comparing two items at a time. However, the trade-off is that pairwise evals take more time and need transformation into pointwise for traditional search metrics or ranking data. Doug suggests mitigating these factors using approaches like LLMs and systems like Elo to convert pairwise comparisons into a pointwise rating.',
    #"Explain vector search in Doug's voice.",
    #"What is Doug's favorite search engine?",
    #"How do I write a Solr plugin?"
}

score_map = {
    'NO-MATCH': 0.0,
    'PARTIAL': 0.5,
    'MATCH': 1.0
}

with_titles = []
for chunk_fns in [chunk_by_token_length, chunk_by_overlap, no_chunking]:
    fn = chunk_with_title(chunk_fns)
    fn.__name__ = chunk_fns.__name__ + "__title"
    with_titles.append(fn)

scores = Counter()

for chunk_fn in [chunk_by_token_length, chunk_by_overlap, no_chunking] + with_titles:
    display(Markdown(f"# ChunkFn: {chunk_fn.__name__}"))

    rag = RAG(corpus, openai, chunk_fn=chunk_fn)
    for question, expected in message_w_answer.items():
        display(Markdown(f"## {question}"))

        answer, query, results = rag.chat(question)
        correct = rag.judge(expected, answer)
        try:
            scores[chunk_fn.__name__] += score_map[correct]
        except KeyError:
            pass 
        display(Markdown(
            f'**Question:** {question}\n\n**Rewritten query:** {query}\n\n**Answer:** {answer}\n\n**Correct?** {correct}'
        ))
        # display(Markdown('### Evidence supplied to the model'))
        # for snippet in results:
        #     title = ""
        #     display(Markdown(f'**{title}**\n\n{snippet}'))
        # display(HTML("</hr>"))

# ChunkFn: chunk_by_token_length

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/194 [00:00<?, ?it/s]

## What is Bayesian BM25?

Rewritten:  Search query: "Bayesian BM25 explanation"


...

Rewritten:  Benefits of search management


**Question:** How is search management useful?

**Rewritten query:** Benefits of search management

**Answer:** Doug observes that in cases where search involves billions of unique queries, long tail queries, and a huge catalog, search management can be useful for focusing on edge cases, head queries, and attributes of ideal products rather than mapping queries directly to products. By vigilantly measuring, reviewing, and retiring manual interventions in the search system, teams can proactively address important issues and ensure the search algorithm stays tuned for optimal performance, watching closely for any deviations. Search management in such scenarios aims at patching key problems efficiently while maintaining a vigilant approach to various aspects of the search system.

**Correct?** PARTIAL

## Why use pairwise evals over pointwise

Rewritten:  Comparison of pairwise evaluations versus pointwise evaluations in machine learning.


**Question:** Why use pairwise evals over pointwise

**Rewritten query:** Comparison of pairwise evaluations versus pointwise evaluations in machine learning.

**Answer:** Doug highlights the advantages of using pairwise evaluations over pointwise ones. Pairwise evaluations, which involve comparing two items at a time for relevance, offer benefits such as less per-decision error, more precise results, and quicker decisions. While pairwise evaluations can be more time-consuming and need to be transformed into pointwise ratings for traditional metrics, mitigation strategies like leveraging LLMs for simpler evaluations or employing systems like Elo can address these challenges, providing a nuanced approach to improving search relevance.

**Correct?** MATCH

In [95]:
scores

Counter({'no_chunking': 3.0,
         'chunk_by_overlap__title': 3.0,
         'chunk_by_token_length': 1.5,
         'chunk_by_overlap': 1.5,
         'chunk_by_token_length__title': 1.5,
         'no_chunking__title': 1.5})

In [102]:
scores

Counter({'chunk_by_token_length__title': 3.5,
         'no_chunking__title': 3.5,
         'chunk_by_overlap__title': 3.0,
         'chunk_by_token_length': 2.5,
         'chunk_by_overlap': 2.0,
         'no_chunking': 1.5})